# 多个 Tool_call in Agent
1. Agent 发送请求后，LLM 返回，同时要求两个 Tool_call：

```js
[
    function_call(
        name="get_user_name",
        call_id="call_A"
    ),

    function_call(
        name="calculator",
        call_id="call_B"
    )
]
```

2. LLM 负责提出行动计划/工具调用请求，Agent Runtime 负责实际执行这些请求。
也就是说，执行 Tool_call 的逻辑不在 LLM 内部，而是由 Agent Runtime 决定

3. 如果两个 Tool_call 之间<font color="Coral">没有依赖关系</font>，LLM 可以同时提出两个 Tool_call 请求

4. 如果两个 Tool_call 之间<font color="Coral">有依赖关系</font>，LLM 必须先提出第一个 Tool_call 请求
，等到 Agent Runtime 执行完后，再提出第二个 Tool_call 请求

现在这个 Agent 架构里：

1）LLM 决定<font color="red">“需要调用哪些 Tool、何时提出下一次 Tool Call、以及 Tool Call 之间的逻辑依赖”；</font>

2）Agent Runtime 决定“这些 Tool Call 实际怎么执行”。
当然，可以写 prompt 让 LLM 去决定“Tool Call 实际怎么执行”

## 输出如下
```shell
--- Processing Response ---
Tool Call: get_user_name
Arguments: {}
Tool Result: Sylvie

--- Processing Response ---
Tool Call: calculator
Arguments: {"a":12345,"b":6789,"operation":"add"}
Tool Result: 19134

--- Processing Response ---
Tool Call: calculator
Arguments: {"a":19134,"b":2,"operation":"multiply"}
Tool Result: 38268

--- Processing Response ---

Final Answer:
你的名字是 Sylvie。
计算结果：先算 12345 + 6789 = 19134，再将结果乘以 2，得到 38268。
```

In [ ]:
import json
import os

from dotenv import load_dotenv
from openai import OpenAI


# =========================
# 1. 加载环境变量
# =========================

load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")
base_url = os.getenv("OPENAI_BASE_URL")

if not api_key:
    raise ValueError("OPENAI_API_KEY not found.")

if not base_url:
    raise ValueError("OPENAI_BASE_URL not found.")


# =========================
# 2. 创建 LLM Client
# =========================

client = OpenAI(
    api_key=api_key,
    base_url=base_url,
)


# =========================
# 3. Tool
# =========================

def calculator(
    a: float,
    b: float,
    operation: str,
) -> float:

    if operation == "add":
        return a + b

    if operation == "subtract":
        return a - b

    if operation == "multiply":
        return a * b

    if operation == "divide":
        return a / b

    raise ValueError(
        f"Unknown operation: {operation}"
    )

def get_user_name() -> str:
    """获取当前用户的名字。"""
    return "Sylvie"

# =========================
# 4. Tool Registry
# =========================

tool_registry = {
    "calculator": calculator,
    "get_user_name": get_user_name,
}


# =========================
# 5. Tool Schema
# =========================

tools = [
    {
        "type": "function",
        "name": "calculator",
        "description": "执行基本数学计算。",
        "parameters": {
            "type": "object",
            "properties": {
                "a": {
                    "type": "number",
                    "description": "第一个数字",
                },
                "b": {
                    "type": "number",
                    "description": "第二个数字",
                },
                "operation": {
                    "type": "string",
                    "enum": [
                        "add",
                        "subtract",
                        "multiply",
                        "divide",
                    ],
                    "description": "计算操作",
                },
            },
            "required": [
                "a",
                "b",
                "operation",
            ],
        },
    },
    {
        "type": "function",
        "name": "get_user_name",
        "description": "获取当前用户的名字。",
        "parameters": {
            "type": "object",
            "properties": {},
            "required": [],
        },
    }
]


# =========================
# 6. 初始任务
# =========================

input_items = [
    {
        "role": "system",
        "content": """
你是一个计算 Agent。

所有数学计算都必须使用 calculator 工具完成，
不能直接在语言模型内部计算。

如果一个任务包含多个计算步骤，
必须按照步骤逐步调用 calculator，
并将前一步的结果作为后一步的输入。
""",
    },
    {
        "role": "user",
        "content": "请告诉我我的名字，然后计算 12345 + 6789，再把结果乘以2。",
    }
]


# =========================
# 7. Agent Loop
# =========================

response = client.responses.create(
    model="gpt-5",
    input=input_items,
    tools=tools,
)

while True:

    print("\n--- Processing Response ---")

    tool_outputs = []

    for item in response.output:

        if item.type == "function_call":

            print("Tool Call:", item.name)
            print("Arguments:", item.arguments)

            tool_name = item.name

            arguments = json.loads(item.arguments)

            # 找到 Python Function
            tool = tool_registry[tool_name]

            # 执行 Tool
            result = tool(**arguments)

            print("Tool Result:", result)

            # 构造 Tool Result，返回结果到 LLM
            tool_outputs.append(
                {
                    "type": "function_call_output",
                    # 对应这次 Tool Call 的 call_id
                    "call_id": item.call_id,
                    "output": str(result),
                }
            )

    # =========================
    # 没有 Tool Call
    # =========================

    if not tool_outputs:

        print("\nFinal Answer:")
        print(response.output_text)

        break

    # =========================
    # 把 Tool Result
    # 交给下一轮 LLM
    # =========================

    response = client.responses.create(
        model="gpt-5",
        # 上一轮response，用于继续处理
        previous_response_id=response.id,
        input=tool_outputs,
        tools=tools,
    )